# 08. 설명력과 "설명 못 한 몫" (한계 정리용)

입력: `data/3_결과/train_table.csv` (05 결과, 07과 같은 표)

**왜 하나**
- 07은 "A가 오르면 사고 오즈가 몇 배"(효과 크기)를 봄
- 08은 "A·날씨가 사고/평소 차이를 **얼마나** 가려내나"(설명력)를 봄
- 남는 몫 = 우리가 못 잰 것(운전자 나이·동공·습관·반응속도, 차량, 우연, 데이터 오차)

**보는 숫자 3개**
1. **의사 결정계수(맥패든 R²)**: 로지스틱용 결정계수. 0이면 동전 던지기, 1이면 완벽
2. **A를 넣었을 때 늘어난 설명력(ΔR²)** + 그게 우연인지 검정(우도비 검정 p)
3. **맞힘 비율**: 같은 묶음 안에서 "사고 난 순간"을 "평소"보다 위험하다고 본 비율. 우연이면 0.5

> 초안 · 업데이트 중 (검토 전). 교통량(VDS)·길이 그룹이 들어가면 숫자는 바뀔 수 있음

In [ ]:
import pandas as pd, numpy as np, warnings
from pathlib import Path
from scipy import stats
from statsmodels.discrete.conditional_models import ConditionalLogit
warnings.filterwarnings('ignore')

DATA = Path('data')
t = pd.read_csv(DATA / '3_결과' / 'train_table.csv', low_memory=False)

# 07과 똑같이 준비
cc = t[t.design.isin(['사고', '같은지점'])].copy()
cc = cc[cc.case_id.isin(cc[cc.y == 0].case_id)]
cc['rain'] = (cc.rn.fillna(0) > 0).astype(int)
cc['vs_km'] = cc.vs / 100
cc['hol'] = cc.is_holiday.astype(int)
print('사고', cc[cc.y == 1].case_id.nunique(), '건 / 평소', (cc.y == 0).sum(), '행')

In [ ]:
def clean(d, cols):
    d = d.dropna(subset=cols)
    d = d[d.groupby('case_id').y.transform('sum') == 1]
    return d[d.groupby('case_id').y.transform('size') > 1]

def ll_null(d):
    # 계수가 전부 0 = 묶음 안 모두 같은 확률 → 묶음마다 log(1/묶음크기)
    return -np.log(d.groupby('case_id').size()).sum()

def fit_ll(d, cols):
    if not cols:
        return ll_null(d), None
    m = ConditionalLogit(d.y, d[cols], groups=d.case_id).fit(disp=0)
    return m.llf, m

def concord(d, m, cols):
    # 묶음 안에서 사고 순간 점수 > 평소 점수 인 쌍의 비율 (같으면 0.5)
    s = d[cols].values @ m.params[cols].values
    d = d.assign(s=s)
    out = []
    for _, g in d.groupby('case_id'):
        sc = g.loc[g.y == 1, 's'].iloc[0]
        sr = g.loc[g.y == 0, 's'].values
        out.append(((sc > sr) + 0.5 * (sc == sr)).mean())
    return float(np.mean(out))

def ladder(d, steps, label):
    '''steps = [(이름, 변수목록), ...] 차례로 변수를 더하며 설명력 증가 확인'''
    allcols = sorted({c for _, cs in steps for c in cs})
    d = clean(d, allcols)
    L0 = ll_null(d)
    rows, prev = [], (L0, 0)
    for name, cols in steps:
        L, m = fit_ll(d, cols)
        lr = 2 * (L - prev[0]); df = len(cols) - prev[1]
        rows.append({'모델': name,
                     '의사R²(맥패든)': round(1 - L / L0, 4),
                     '늘어난 R²': round((L - prev[0]) / -L0, 4),
                     '우도비 p(앞 단계 대비)': round(stats.chi2.sf(lr, df), 3) if df > 0 else None,
                     '맞힘 비율(우연 0.5)': round(concord(d, m, cols), 3) if m is not None else 0.5,
                     '설명 못 한 몫': round(L / L0, 4)})
        prev = (L, len(cols))
    print(f'== {label}: 사고 {d.case_id.nunique()}건, 묶음당 평균 {len(d)/d.case_id.nunique():.1f}행')
    return pd.DataFrame(rows), d

## 1. 저울 점검: 전체 사고에서 날씨만으로 얼마나 설명되나

날씨(비·시정·공휴일)는 효과가 확실하다고 알려진 변수. 이것도 R²가 작게 나오면,
"사고는 원래 우연·운전자 몫이 커서 R²가 작다"는 근거가 됨 (A만의 문제가 아님).

In [ ]:
tab, _ = ladder(cc, [('날씨만', ['rain', 'vs_km', 'hol'])], '전체 (방법 점검)')
tab

## 2. 핵심: A를 더하면 설명력이 얼마나 느나 (낮 사고)

차례: 아무것도 없음(0) → 비 → 비 + A
- "늘어난 R²"의 마지막 줄 = **A가 보탠 설명력**
- 우도비 p < 0.05 이면 A를 넣은 게 우연 이상으로 나아졌다는 뜻

In [ ]:
res = {}
for zones, col, lab in [(['진입전', '경계부'], 'A_진입', '진입'),
                        (['출구부', '진출후'], 'A_진출', '진출')]:
    z = cc[cc.tunnel_zone.isin(zones)]
    day_cases = z[(z.y == 1) & (z.sun_elev > 0)].case_id
    d = z[z.case_id.isin(day_cases)].copy()
    d['A10'] = d[col] * 10
    res[lab], _ = ladder(d, [('비만', ['rain']), ('비 + A', ['rain', 'A10'])], f'{lab} 낮')
    display(res[lab])

## 3. 숫자가 얼마나 흔들리나 (사고를 다시 뽑아 100번 반복)

사고 수가 적어서 R²도 흔들림. 사고 묶음을 복원추출로 100번 다시 뽑아
"A가 보탠 R²"의 95% 범위를 봄. 범위 아래쪽이 0 근처면 "A의 설명력은 아직 불확실".

In [ ]:
def boot_dr2(d, base, full, n=100, seed=0):
    d = clean(d, full)
    ids = d.case_id.unique(); rng = np.random.default_rng(seed)
    g = {k: v for k, v in d.groupby('case_id')}
    out = []
    for b in range(n):
        pick = rng.choice(ids, len(ids), replace=True)
        bd = pd.concat([g[k].assign(case_id=f'{k}_{i}') for i, k in enumerate(pick)])
        try:
            L0 = ll_null(bd); L1, _ = fit_ll(bd, base); L2, _ = fit_ll(bd, full)
            out.append((L2 - L1) / -L0)
        except Exception:
            pass
    q = np.percentile(out, [2.5, 50, 97.5])
    return pd.Series({'A가 보탠 R² 중앙값': round(q[1], 4), '95%_하한': round(q[0], 4),
                      '95%_상한': round(q[2], 4), '성공 반복': len(out)})

for zones, col, lab in [(['진입전', '경계부'], 'A_진입', '진입'),
                        (['출구부', '진출후'], 'A_진출', '진출')]:
    z = cc[cc.tunnel_zone.isin(zones)]
    day_cases = z[(z.y == 1) & (z.sun_elev > 0)].case_id
    d = z[z.case_id.isin(day_cases)].copy()
    d['A10'] = d[col] * 10
    print(lab); display(boot_dr2(d, ['rain'], ['rain', 'A10']))

## 4. 못 잰 요인이 결과를 뒤집으려면 얼마나 강해야 하나 (E값)

운전자 나이·습관 같은 못 잰 요인이 A 효과를 **가짜로 만들려면**,
그 요인이 (1) A가 높은 날에 더 많이 몰리고 (2) 사고 위험도 올려야 함. 둘 다 최소 몇 배여야 하는지가 E값.
- E값 2.1 = "둘 다 2.1배 이상인 숨은 요인이 있어야 A 효과가 사라짐"
- 범위 하한의 E값이 1 = 지금도 범위가 1을 포함 → 약한 요인으로도 설명될 수 있음 (= "경향" 수준이라는 뜻)
- 사고는 드물어서 오즈비 ≈ 위험비로 보고 계산 (VanderWeele·Ding 2017)

In [ ]:
def e_value(rr):
    rr = 1 / rr if rr < 1 else rr
    return rr + np.sqrt(rr * (rr - 1))

rows = []
for zones, col, lab in [(['진입전', '경계부'], 'A_진입', '진입'),
                        (['출구부', '진출후'], 'A_진출', '진출')]:
    z = cc[cc.tunnel_zone.isin(zones)]
    day_cases = z[(z.y == 1) & (z.sun_elev > 0)].case_id
    d = z[z.case_id.isin(day_cases)].copy()
    d['A10'] = d[col] * 10
    d = clean(d, ['A10', 'rain'])
    m = ConditionalLogit(d.y, d[['A10', 'rain']], groups=d.case_id).fit(disp=0)
    orr = np.exp(m.params.A10); lo, hi = np.exp(m.conf_int().loc['A10'])
    near = 1.0 if lo <= 1 <= hi else (lo if orr > 1 else hi)
    rows.append({'구간': lab, 'A 0.1당 배수': round(orr, 2), '95%': f'{lo:.2f}~{hi:.2f}',
                 'E값(추정치)': round(e_value(orr), 2), 'E값(범위 하한)': round(e_value(near), 2)})
pd.DataFrame(rows)

## 5. (있으면) 사고원인별로 A 효과가 다른가

운전자 쪽 정보는 없지만 `사고원인`(예: 주시태만, 졸음, 과속)이 **거친 대리 변수**가 됨.
눈이 못 따라가서 생긴 사고라면 A 효과는 "주시태만·전방주시" 쪽에서 더 커야 하고, 졸음에서는 약해야 함.
`train_table.csv`에 사고원인 열이 없으면 이 칸은 건너뜀. 원인별 사고 수가 적으면 범위가 넓으니 "방향만" 봄.

In [ ]:
if '사고원인' not in cc.columns:
    print('train_table.csv에 사고원인 열이 없음 → 05에서 사고원인을 붙이면 실행 가능')
else:
    z = cc[cc.tunnel_zone.isin(['진입전', '경계부'])].copy()
    cause = z[z.y == 1].set_index('case_id')['사고원인']
    z['원인'] = z.case_id.map(cause)
    print(z[z.y == 1].groupby('원인').size().sort_values(ascending=False).head(10))
    for c in z[z.y == 1]['원인'].value_counts().index[:4]:
        d = z[(z['원인'] == c) & z.case_id.isin(z[(z.y == 1) & (z.sun_elev > 0)].case_id)].copy()
        d['A10'] = d['A_진입'] * 10
        try:
            dd = clean(d, ['A10', 'rain'])
            m = ConditionalLogit(dd.y, dd[['A10', 'rain']], groups=dd.case_id).fit(disp=0)
            ci = np.exp(m.conf_int().loc['A10'])
            print(f'{c}: 사고 {dd.case_id.nunique()}건, A 0.1당 {np.exp(m.params.A10):.2f}배 ({ci[0]:.2f}~{ci[1]:.2f})')
        except Exception as e:
            print(f'{c}: 사고가 너무 적어 추정 불가')

## 6. 읽는 법 메모

- 사고 자료에서 맥패든 R²가 0.01~0.05처럼 **작게 나오는 게 정상**. 같은 자리·같은 시각에 수천 대가 지나가도 사고는 한두 건이라, 누가 사고를 내는지는 대부분 운전자·우연 몫
- R²가 작아도 A 효과(배수)가 틀렸다는 뜻은 아님. "A가 위험을 올리긴 하지만, 사고를 혼자 다 설명하지는 못한다"로 읽음
- "설명 못 한 몫"에는 운전자 요인만 있는 게 아님: 차량, 교통 흐름(VDS 들어가기 전), 시각·위치 오차, 순수 우연이 다 섞여 있음 → **운전자 몫만 따로 떼어 숫자로 말할 수는 없음**